# BYOM (Bring Your Own Metrics)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/your_own_metrics.ipynb)

This notebook demonstrates how to create custom metrics by subclassing `BaseMetric` and `ThresholdMetric`. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import xarray as xr
import extremeweatherbench as ewb
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: 2022 Southern Plains Heat Wave — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9004,
    title="2022 Southern Plains Heat Wave (demo)",
    start_date=datetime.datetime(2022, 7, 19),
    end_date=datetime.datetime(2022, 7, 22),
    location=BoundingBoxRegion.create_region(
        latitude_min=31.0,
        latitude_max=37.0,
        longitude_min=260.0,
        longitude_max=266.0,
    ),
    event_type="heat_wave",
)
cases = [demo_case]

In [ ]:
class MeanAbsolutePercentageError(ewb.BaseMetric):
    """Mean Absolute Percentage Error."""

    def __init__(self, name: str = "MAPE", **kwargs):
        super().__init__(name=name, **kwargs)

    def _compute_metric(
        self,
        forecast: xr.DataArray,
        target: xr.DataArray,
        **kwargs,
    ) -> xr.DataArray:
        percentage_error = (
            (forecast - target).abs()
            / target.where(target != 0)
        ) * 100
        return percentage_error.mean(
            dim=[
                d
                for d in percentage_error.dims
                if d != self.preserve_dims
            ]
        )


class ProbabilityOfDetection(ewb.ThresholdMetric):
    """Probability of Detection (Hit Rate)."""

    def __init__(
        self, name: str = "ProbabilityOfDetection", **kwargs
    ):
        super().__init__(name=name, **kwargs)

    def _compute_metric(
        self,
        forecast: xr.DataArray,
        target: xr.DataArray,
        **kwargs,
    ):
        transformed = kwargs.get("transformed_manager")
        if transformed is None:
            transformed = self.transformed_contingency_manager(
                forecast=forecast,
                target=target,
                forecast_threshold=self.forecast_threshold,
                target_threshold=self.target_threshold,
                preserve_dims=self.preserve_dims,
            )
        counts = transformed.get_counts()
        tp = counts["tp_count"]
        fn = counts["fn_count"]
        return tp / (tp + fn)

In [ ]:
mape = MeanAbsolutePercentageError(
    forecast_variable="surface_air_temperature",
    target_variable="surface_air_temperature",
)

pod = ProbabilityOfDetection(
    forecast_variable="surface_air_temperature",
    target_variable="surface_air_temperature",
    forecast_threshold=308.15,
    target_threshold=308.15,
)

forecast = ewb.ZarrForecast(
    source="gs://weatherbench2/datasets/hres/2016-2022-0012-1440x721.zarr",
    name="HRES",
    variable_mapping=ewb.HRES_metadata_variable_mapping,
    storage_options={"remote_options": {"anon": True}},
)

target = ewb.ERA5(variables=["surface_air_temperature"])

eval_objects = [
    ewb.EvaluationObject(
        event_type="heat_wave",
        metric_list=[mape, pod],
        target=target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()
print(outputs)